In [6]:
from pathlib import Path
from collections import Counter
from pprint import pprint

import hashlib
import json
import re
import unicodedata

import pandas as pd
from tqdm.auto import tqdm
from datasets import load_dataset

In [7]:
PROJECT_ROOT = Path(
    r"D:\dev\projects\fourlang_translation"
)

# 原始/来源信息
RAW_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "en_uz"
    / "hplt"
)

# 清洗结果
CLEAN_DIR = (
    PROJECT_ROOT
    / "data"
    / "clean"
    / "en_uz"
    / "hplt"
)

# 最终人工/合规确认之后再进入这里
APPROVED_DIR = (
    PROJECT_ROOT
    / "data"
    / "approved"
    / "en_uz"
)

# Benchmark
BENCHMARK_DIR = (
    PROJECT_ROOT
    / "data"
    / "benchmark"
)

DOCS_DIR = PROJECT_ROOT / "docs"

RESULT_DIR = PROJECT_ROOT / "results"

for path in [
    RAW_DIR,
    CLEAN_DIR,
    APPROVED_DIR,
    BENCHMARK_DIR,
    DOCS_DIR,
    RESULT_DIR,
]:
    path.mkdir(
        parents=True,
        exist_ok=True
    )

print("PROJECT_ROOT :", PROJECT_ROOT)
print("RAW_DIR      :", RAW_DIR)
print("CLEAN_DIR    :", CLEAN_DIR)

PROJECT_ROOT : D:\dev\projects\fourlang_translation
RAW_DIR      : D:\dev\projects\fourlang_translation\data\raw\en_uz\hplt
CLEAN_DIR    : D:\dev\projects\fourlang_translation\data\clean\en_uz\hplt


In [8]:
DATASET_ID = "HPLT/DocHPLT"

CONFIG_NAME = "en-uz"

REVISION = (
    "1c94ae042138c768caccb3cbc27f5d974a66ea0f"
)

SEED = 42

In [9]:
# 第一轮只抽10K
TARGET_PAIRS = 10_000

# 最多扫描多少文档，防止意外无限跑
MAX_DOCS_TO_SCAN = 50_000

# Streaming shuffle buffer
SHUFFLE_BUFFER = 2_000

In [10]:
def load_hplt_stream(
    shuffle=False
):
    dataset = load_dataset(
        DATASET_ID,
        CONFIG_NAME,
        split="train",
        streaming=True,
        revision=REVISION,
    )

    if shuffle:
        dataset = dataset.shuffle(
            seed=SEED,
            buffer_size=SHUFFLE_BUFFER,
        )

    return dataset

In [11]:
dataset = load_hplt_stream()

print(dataset)

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/45 [00:00<?, ?it/s]

IterableDataset({
    features: ['src_doc_id', 'tgt_doc_id', 'lang_pair', 'src_doc', 'tgt_doc', 'alignment'],
    num_shards: 6
})


In [12]:
sample = next(
    iter(dataset)
)

print(
    sample.keys()
)

dict_keys(['src_doc_id', 'tgt_doc_id', 'lang_pair', 'src_doc', 'tgt_doc', 'alignment'])


In [13]:
print(
    "lang_pair:",
    sample["lang_pair"]
)

lang_pair: en-uz


In [14]:
print("=== SRC DOC ===")

print(
    "ID数量:",
    len(sample["src_doc"]["ids"])
)

print(
    "句子数量:",
    len(sample["src_doc"]["sentences"])
)

for sid, sentence in zip(
    sample["src_doc"]["ids"][:10],
    sample["src_doc"]["sentences"][:10],
):
    print(
        sid,
        "=>",
        sentence
    )

=== SRC DOC ===
ID数量: 13
句子数量: 13
1.1 => What do these people do for a living?
1.2 => Of the three, which one has been convicted of a crime?
2.1 => Before you look at their information below, take a good, long look at their photos, hoodie and all.
3.1 => [spoiler title=”see if your perception is correct”]This is Noel Williams.
3.2 => He is an Information Technology Specialist with a degree from Wilberforce University.
3.3 => Mr. Williams has no criminal record.[/spoiler]
4.1 => [spoiler title=”see if your perception is correct”]This is Jay Gatsby.
4.2 => He is a journalist, podcaster, philanthropist and father.
4.3 => Mr. Gatsby has no criminal record.[/spoiler]
5.1 => [spoiler title=”see if your perception is correct”]This is medical doctor, Renee Matthews.


In [15]:
print("=== TGT DOC ===")

print(
    "ID数量:",
    len(sample["tgt_doc"]["ids"])
)

print(
    "句子数量:",
    len(sample["tgt_doc"]["sentences"])
)

for sid, sentence in zip(
    sample["tgt_doc"]["ids"][:10],
    sample["tgt_doc"]["sentences"][:10],
):
    print(
        sid,
        "=>",
        sentence
    )

=== TGT DOC ===
ID数量: 86
句子数量: 86
1.1 => 1. ''Yuksak ma’naviyat – yengilmas kuch'' asarining tuzilishi aytib bering?[spoiler]Asar muqaddima 4 ta bob, 10 fasl, xotimadan iborat [/spoiler] 2. ''Yuksak ma’naviyat – yengilmas kuch'' asarining birinchi bobi qanday nomlanadi?[spoiler]Ma’naviyat insonning ulg`ayishi va kuch qudrati manbaidir.[/spoiler] 3. ''Yuksak ma’naviyat – yengilmas kuch'' asarining ikkinchi bobi qanday nomlanadi?[spoiler]Mustaqillik – ma’naviy tiklanish va yuksalish[/spoiler] 4. ''Yuksak ma’naviyat – yengilmas kuch'' asarining uchinchi bobi qanday nomlanadi?[spoiler]Ma’naviyatga tahdid – o’zligimiz va kelajagimizga tahdid [/spoiler] 5. ''Yuksak ma’naviyat – yengilmas kuch'' asarining birinchi bobining fasllari qanday nomlanadi?[spoiler]Ma’naviyatni anglash, Ma’naviyatni shakllantiradigan asosiy mezonlar, Ma’naviy va moddiy hayot uyg`unligi[/spoiler] 6.
1.2 => Milliy ma’naviyatimizni shakllantiradigan asosiy omillarini ayting[spoiler]Ma’naviy meros, madaniy boyliklar, ko’

In [16]:
print("=== ALIGNMENT ===")

for item in sample["alignment"][:10]:
    pprint(item)

=== ALIGNMENT ===
{'aligner-score': 0.24569900333881378,
 'bicleaner-score': 0.0,
 'bifixer-score': 0.9369999766349792,
 'src': ['3.3'],
 'tgt': ['1.54']}
{'aligner-score': 0.20424999296665192,
 'bicleaner-score': 0.0,
 'bifixer-score': 0.9398999810218811,
 'src': ['5.4'],
 'tgt': ['1.68']}


In [17]:
def collect_score_sample(
    limit=5_000
):
    dataset = load_hplt_stream(
        shuffle=False
    )

    rows = []

    for document in tqdm(
        dataset,
        desc="Collecting score sample"
    ):
        alignments = (
            document.get("alignment")
            or []
        )

        for item in alignments:

            src_ids = (
                item.get("src")
                or []
            )

            tgt_ids = (
                item.get("tgt")
                or []
            )

            # 只研究一对一句
            if (
                len(src_ids) != 1
                or len(tgt_ids) != 1
            ):
                continue

            rows.append({
                "aligner_score":
                    item.get(
                        "aligner-score"
                    ),

                "bicleaner_score":
                    item.get(
                        "bicleaner-score"
                    ),

                "bifixer_score":
                    item.get(
                        "bifixer-score"
                    ),
            })

            if len(rows) >= limit:
                return pd.DataFrame(
                    rows
                )

    return pd.DataFrame(rows)

In [18]:
score_df = collect_score_sample(
    limit=5_000
)

score_df.describe(
    percentiles=[
        0.1,
        0.25,
        0.5,
        0.75,
        0.9,
        0.95,
    ]
)

Resolving data files:   0%|          | 0/45 [00:00<?, ?it/s]

,aligner_score,bicleaner_score,bifixer_score
count,5000.000000,5000.000000,5000.000000
mean,0.501447,0.439187,1.056481
std,0.206122,0.474151,1.211862
min,0.200002,0.000000,0.628500
10%,0.267393,0.000000,0.872170
25%,0.344058,0.000000,0.909800
50%,0.462475,0.013500,0.932500
75%,0.605651,0.996000,0.951925
90%,0.786014,0.999000,0.993800
95%,1.000000,1.000000,1.751645


In [19]:
ALIGNER_MIN = 0.50

BICLEANER_MIN = 0.80

In [20]:
MIN_CHARS = 2

MAX_CHARS = 300

MIN_LENGTH_RATIO = 0.30

MAX_LENGTH_RATIO = 3.00

In [21]:
def clean_text(text):
    if text is None:
        return ""

    text = str(text)

    text = unicodedata.normalize(
        "NFKC",
        text
    )

    # 合并多余空白
    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()

In [22]:
def normalize_for_match(text):
    text = clean_text(text)

    text = text.casefold()

    # 去掉标点
    text = re.sub(
        r"[^\w\s]",
        "",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()

In [23]:
CYRILLIC_PATTERN = re.compile(
    r"[\u0400-\u04FF]"
)


def has_cyrillic(text):
    return bool(
        CYRILLIC_PATTERN.search(
            str(text)
        )
    )

In [24]:
print(
    has_cyrillic(
        "Men talabaman."
    )
)

print(
    has_cyrillic(
        "Мен талабаман."
    )
)

False
True


In [25]:
def looks_like_markup(text):
    text_lower = text.lower()

    bad_patterns = [
        "<html",
        "</html",
        "<script",
        "</script",
        "<style",
        "</style",
        "{javascript",
    ]

    return any(
        pattern in text_lower
        for pattern in bad_patterns
    )

In [26]:
def valid_basic_text(
    en_text,
    uz_text,
):
    en_text = clean_text(en_text)
    uz_text = clean_text(uz_text)

    if not en_text or not uz_text:
        return False

    if (
        len(en_text) < MIN_CHARS
        or len(uz_text) < MIN_CHARS
    ):
        return False

    if (
        len(en_text) > MAX_CHARS
        or len(uz_text) > MAX_CHARS
    ):
        return False

    if looks_like_markup(en_text):
        return False

    if looks_like_markup(uz_text):
        return False

    # Uzbek第一版只保留Latin
    if has_cyrillic(uz_text):
        return False

    ratio = (
        len(en_text)
        / max(
            len(uz_text),
            1
        )
    )

    if not (
        MIN_LENGTH_RATIO
        <= ratio
        <= MAX_LENGTH_RATIO
    ):
        return False

    # 两边完全一样，一般值得怀疑
    if (
        normalize_for_match(en_text)
        ==
        normalize_for_match(uz_text)
    ):
        return False

    return True

In [27]:
UZ_BENCHMARK = [
    "Salom.",
    "Rahmat.",
    "Men talabaman.",
    "Bugun havo yaxshi.",
    "Men Toshkentda yashayman.",
    "Men ertaga ishga boraman.",
    "Men ertaga aeroportga boraman.",
    (
        "Men ertaga ertalab "
        "soat sakkizda "
        "aeroportga boraman."
    ),
]

In [28]:
EN_BENCHMARK = [
    "Hello.",
    "Thank you.",
    "I am a student.",
    "The weather is good today.",
    "I live in Tashkent.",
    "I will go to work tomorrow.",
    "I will go to the airport tomorrow.",
    (
        "I will go to the airport "
        "at eight tomorrow morning."
    ),
]

In [29]:
UZ_BENCHMARK_NORMALIZED = {
    normalize_for_match(x)
    for x in UZ_BENCHMARK
}

EN_BENCHMARK_NORMALIZED = {
    normalize_for_match(x)
    for x in EN_BENCHMARK
}

In [30]:
{
    "ids": [
        "1.1",
        "1.2"
    ],

    "sentences": [
        "...",
        "..."
    ]
}

{'ids': ['1.1', '1.2'], 'sentences': ['...', '...']}

In [31]:
def make_sentence_map(doc):
    ids = (
        doc.get("ids")
        or []
    )

    sentences = (
        doc.get("sentences")
        or []
    )

    return {
        sentence_id: sentence
        for sentence_id, sentence
        in zip(
            ids,
            sentences
        )
    }

In [32]:
def safe_float(
    value,
    default=0.0
):
    try:
        if value is None:
            return default

        return float(value)

    except (
        TypeError,
        ValueError,
    ):
        return default

In [33]:
def collect_high_quality_pairs(
    target_pairs=TARGET_PAIRS,
    max_docs=MAX_DOCS_TO_SCAN,
):

    dataset = load_hplt_stream(
        shuffle=True
    )

    accepted = []

    seen_pairs = set()

    stats = Counter()

    for doc_index, document in enumerate(
        tqdm(
            dataset,
            desc="Scanning DocHPLT"
        )
    ):

        if doc_index >= max_docs:
            break

        stats["documents_scanned"] += 1

        lang_pair = document.get(
            "lang_pair",
            ""
        )

        # 理论上subset就是en-uz
        # 这里仍做保护
        if lang_pair not in [
            "en-uz",
            "uz-en",
        ]:
            stats[
                "reject_wrong_lang_pair"
            ] += 1
            continue

        src_map = make_sentence_map(
            document["src_doc"]
        )

        tgt_map = make_sentence_map(
            document["tgt_doc"]
        )

        alignments = (
            document.get("alignment")
            or []
        )

        for alignment in alignments:

            stats[
                "alignments_scanned"
            ] += 1

            src_ids = (
                alignment.get("src")
                or []
            )

            tgt_ids = (
                alignment.get("tgt")
                or []
            )

            # ----------------------------
            # 只保留1 ↔ 1
            # ----------------------------

            if (
                len(src_ids) != 1
                or len(tgt_ids) != 1
            ):
                stats[
                    "reject_not_1to1"
                ] += 1
                continue

            src_id = src_ids[0]
            tgt_id = tgt_ids[0]

            src_text = src_map.get(
                src_id
            )

            tgt_text = tgt_map.get(
                tgt_id
            )

            if (
                src_text is None
                or tgt_text is None
            ):
                stats[
                    "reject_missing_sentence"
                ] += 1
                continue

            # ----------------------------
            # 根据lang_pair决定哪边是en / uz
            # ----------------------------

            if lang_pair == "en-uz":

                en_text = clean_text(
                    src_text
                )

                uz_text = clean_text(
                    tgt_text
                )

                en_sentence_id = src_id
                uz_sentence_id = tgt_id

            else:

                en_text = clean_text(
                    tgt_text
                )

                uz_text = clean_text(
                    src_text
                )

                en_sentence_id = tgt_id
                uz_sentence_id = src_id

            # ----------------------------
            # Alignment scores
            # ----------------------------

            aligner_score = safe_float(
                alignment.get(
                    "aligner-score"
                )
            )

            bicleaner_score = safe_float(
                alignment.get(
                    "bicleaner-score"
                )
            )

            bifixer_score = safe_float(
                alignment.get(
                    "bifixer-score"
                )
            )

            if (
                aligner_score
                < ALIGNER_MIN
            ):
                stats[
                    "reject_aligner_score"
                ] += 1
                continue

            if (
                bicleaner_score
                < BICLEANER_MIN
            ):
                stats[
                    "reject_bicleaner_score"
                ] += 1
                continue

            # ----------------------------
            # 基础文本过滤
            # ----------------------------

            if not valid_basic_text(
                en_text,
                uz_text,
            ):
                stats[
                    "reject_basic_text"
                ] += 1
                continue

            # ----------------------------
            # Benchmark leakage
            # ----------------------------

            en_norm = normalize_for_match(
                en_text
            )

            uz_norm = normalize_for_match(
                uz_text
            )

            if (
                en_norm
                in EN_BENCHMARK_NORMALIZED
                or
                uz_norm
                in UZ_BENCHMARK_NORMALIZED
            ):
                stats[
                    "reject_benchmark_leakage"
                ] += 1
                continue

            # ----------------------------
            # 去重
            # ----------------------------

            pair_key = (
                en_norm,
                uz_norm,
            )

            if pair_key in seen_pairs:
                stats[
                    "reject_duplicate"
                ] += 1
                continue

            seen_pairs.add(
                pair_key
            )

            # ----------------------------
            # 保存
            # ----------------------------

            record = {
                "pair_id":
                    f"hplt_{len(accepted):08d}",

                "en":
                    en_text,

                "uz":
                    uz_text,

                "en_sentence_id":
                    en_sentence_id,

                "uz_sentence_id":
                    uz_sentence_id,

                "src_doc_id":
                    document.get(
                        "src_doc_id"
                    ),

                "tgt_doc_id":
                    document.get(
                        "tgt_doc_id"
                    ),

                "lang_pair":
                    lang_pair,

                "aligner_score":
                    aligner_score,

                "bicleaner_score":
                    bicleaner_score,

                "bifixer_score":
                    bifixer_score,

                "dataset":
                    DATASET_ID,

                "dataset_revision":
                    REVISION,
            }

            accepted.append(
                record
            )

            stats["accepted"] += 1

            if (
                len(accepted)
                >= target_pairs
            ):
                return (
                    pd.DataFrame(
                        accepted
                    ),
                    stats,
                )

    return (
        pd.DataFrame(
            accepted
        ),
        stats,
    )

In [34]:
pairs_df, extraction_stats = (
    collect_high_quality_pairs()
)

Resolving data files:   0%|          | 0/45 [00:00<?, ?it/s]

Scanning DocHPLT: 0it [00:00, ?it/s]

In [35]:
print(
    "最终抽取数量:",
    len(pairs_df)
)

print()
print("=== Extraction Statistics ===")

for key, value in (
    extraction_stats
    .most_common()
):
    print(
        f"{key:35s}",
        value
    )

最终抽取数量: 10000

=== Extraction Statistics ===
alignments_scanned                  112891
reject_aligner_score                48547
reject_bicleaner_score              28961
reject_not_1to1                     19943
accepted                            10000
documents_scanned                   5437
reject_duplicate                    4928
reject_basic_text                   512


In [36]:
pd.set_option(
    "display.max_colwidth",
    None
)

pd.set_option(
    "display.max_columns",
    None
)

pairs_df[
    [
        "en",
        "uz",
        "aligner_score",
        "bicleaner_score",
    ]
].head(20)

,en,uz,aligner_score,bicleaner_score
0,"Good place, where you can wait for the, that will never happen.","Yaxshi joy, qaerda kutish mumkin, bu hech qachon bo'lmaydi.",0.632494,0.999
1,- Faculty of Engineering,- Quruvchilik fakulteti,0.549205,0.980
2,- They live mainly in woodland and forest regions.,- Ular asosan o'rmon va o'rmon mintaqalarida istiqomat qilishadi.,0.836660,0.987
3,"However, species can be found in desert regions.","Biroq, turlarni cho'l hududlarida topish mumkin.",0.836660,0.999
4,- They have long tails and slender bodies.,- Ularning uzun quyruqlari va ingichka tanalari bor.,0.816497,0.994
5,- Lacerticles feed mainly on insects.,- Lacerticles asosan hasharotlar bilan oziqlanadi.,1.000000,0.993
6,Some species eat seeds.,Ba'zi turlari urug'larni eyishadi.,0.708073,0.999
7,"AAAA is widely recognized by city, contea, state and federal officials as well as by civic organizations.","AAAA keng shahar tomonidan e'tirof etilgan, tuman, davlat va federal mansabdor shaxslar, shuningdek tomonidan fuqarolik tashkilotlar.",0.696947,0.998
8,"Membership in this Association is open to any person, regardless of citizenship, race, religion, sex, nationality or country of origin.","Bu Assotsiatsiyaga a'zolik har qanday kishi uchun ochiq, qat'i nazar fuqaroligini, irq, din, jins, millati yoki kelib chiqishi mamlakat.",0.599150,0.999
9,- To promote social and cultural events in order to share our ethnic heritage with other Americans and to foster public awareness of our cultural diversity.,- Boshqa amerikaliklar bilan etnik merosini baham uchun ijtimoiy va madaniy tadbirlarni qo'llab-quvvatlash uchun va bizning madaniy xilma-xilligi ijtimoiy ongi da'vat qilish.,0.569906,0.999


In [37]:
print(
    "数据数量:",
    len(pairs_df)
)

print(
    "完全重复pair:",
    pairs_df.duplicated(
        subset=[
            "en",
            "uz",
        ]
    ).sum()
)

数据数量: 10000
完全重复pair: 0


In [38]:
print(
    pairs_df[
        ["en", "uz"]
    ]
    .isna()
    .sum()
)

en    0
uz    0
dtype: int64


In [39]:
pairs_df["en_chars"] = (
    pairs_df["en"].str.len()
)

pairs_df["uz_chars"] = (
    pairs_df["uz"].str.len()
)

pairs_df[
    [
        "en_chars",
        "uz_chars",
        "aligner_score",
        "bicleaner_score",
    ]
].describe()

,en_chars,uz_chars,aligner_score,bicleaner_score
count,10000.000000,10000.000000,10000.000000,10000.000000
mean,94.066900,103.475900,0.645378,0.982706
std,51.624255,55.593964,0.122493,0.034294
min,4.000000,7.000000,0.500000,0.800000
25%,55.000000,61.000000,0.550481,0.984000
50%,86.000000,95.000000,0.615886,0.998000
75%,123.000000,135.000000,0.707107,1.000000
max,300.000000,299.000000,1.000000,1.000000


In [40]:
pairs_df[
    "uz_has_cyrillic"
] = pairs_df[
    "uz"
].apply(
    has_cyrillic
)

pairs_df[
    "uz_has_cyrillic"
].value_counts()

uz_has_cyrillic
False    10000
Name: count, dtype: int64

In [41]:
pairs_df[
    "en_normalized"
] = pairs_df[
    "en"
].apply(
    normalize_for_match
)

pairs_df[
    "uz_normalized"
] = pairs_df[
    "uz"
].apply(
    normalize_for_match
)

In [42]:
leak_df = pairs_df[
    pairs_df[
        "uz_normalized"
    ].isin(
        UZ_BENCHMARK_NORMALIZED
    )
    |
    pairs_df[
        "en_normalized"
    ].isin(
        EN_BENCHMARK_NORMALIZED
    )
]

print(
    "Benchmark leakage:",
    len(leak_df)
)

Benchmark leakage: 0


In [43]:
review_df = (
    pairs_df.sample(
        n=min(
            50,
            len(pairs_df)
        ),
        random_state=SEED,
    )
    [
        [
            "en",
            "uz",
            "aligner_score",
            "bicleaner_score",
        ]
    ]
    .copy()
)

review_df[
    "manual_review"
] = ""

review_df

,en,uz,aligner_score,bicleaner_score,manual_review
6252,"Even have this travel map, so I think some of you might be traveling, moving, taking a journey like this.","Shuningdek, ushbu sayohat xaritasiga ega bo'ling, shuning uchun sizlardan ba'zilari sayohat qilish, harakat qilish va sayohat qilishlari mumkin deb o'ylayman.",0.502046,1.000,
4684,Successful models were developed to explain the interiors of stars and stellar evolution.,"Globula ham siqilib, zichlik oshgach, gravitatsiyaviy energiya issiqlikka aylanib, harorat koʻtariladi.",0.511797,0.961,
1731,The insurance indemnity is paid directly to the victim or beneficiary no later than five working days from the date of signing the Act on Insurance Event.,Sugʻurta qoplamasi bevosita jabrlanuvchiga yoki Naf oluvchiga Sugʻurta hodisasi toʻgʻrisidagi dalolatnoma imzolangan kundan boshlab besh ish kunidan kech boʻlmagan muddatda toʻlab beriladi.,0.533335,0.996,
4742,"Just like Leo, they were born to be in the spotlight.","Xuddi Leo singari, ular ham diqqat markazida bo'lish uchun tug'ilishgan.",0.751140,1.000,
4521,It is used mainly for UI and UX design of web and mobile apps.,U asosan veb va mobil ilovalarning UI va UX dizaynida ishlatiladi.,0.658313,0.991,
6340,"In the File name box, enter a file name for the video, browse to the folder where you want to save the file, and click Save.","Fayl nomi maydoniga video uchun fayl nomini kiriting, faylni saqlamoqchi bo'lgan papkani ko'rib chiqing va Saqlash tugmachasini bosing.",0.640982,1.000,
576,Learn the history and secrets of the thanks to our blackjack online guide France.,Bizning Blackjack onlayn qo'llanma France minnatdorchilik tarixi va sirlarini bilish.,0.609487,0.996,
5202,They love to have the best of everything.,Ular hamma narsaning eng yaxshisini egallashni yaxshi ko'radilar.,0.623610,0.999,
6363,"For example: we go out to work, park our car in the huge parking lot, and then walk to the destination.","Masalan: ishga chiqamiz, mashinamizni ulkan to‘xtash joyiga qo‘yamiz, so‘ng manzilga piyoda boramiz.",0.511414,0.989,
439,"- With the enzyme oriented towards the interior of the cell, the carrier has a high affinity for sodium ions.","- Ferment hujayraning ichki qismiga yo'naltirilganligi sababli, tashuvchi natriy ionlari uchun yuqori yaqinlikka ega.",0.517463,1.000,


In [44]:
REVIEW_FILE = (
    CLEAN_DIR
    / "hplt_en_uz_manual_review_50.csv"
)

review_df.to_csv(
    REVIEW_FILE,
    index=False,
    encoding="utf-8-sig",
)

print(
    REVIEW_FILE
)

D:\dev\projects\fourlang_translation\data\clean\en_uz\hplt\hplt_en_uz_manual_review_50.csv


In [45]:
save_df = pairs_df.drop(
    columns=[
        "uz_has_cyrillic",
        "en_normalized",
        "uz_normalized",
    ],
    errors="ignore",
).copy()

In [46]:
CANDIDATE_CSV = (
    CLEAN_DIR
    / "hplt_en_uz_10k_clean.csv"
)

save_df.to_csv(
    CANDIDATE_CSV,
    index=False,
    encoding="utf-8-sig",
)

In [47]:
CANDIDATE_JSONL = (
    CLEAN_DIR
    / "hplt_en_uz_10k_clean.jsonl"
)

save_df.to_json(
    CANDIDATE_JSONL,
    orient="records",
    lines=True,
    force_ascii=False,
)

In [48]:
CANDIDATE_JSONL = (
    CLEAN_DIR
    / "hplt_en_uz_10k_clean.jsonl"
)

save_df.to_json(
    CANDIDATE_JSONL,
    orient="records",
    lines=True,
    force_ascii=False,
)

In [49]:
print(
    "CSV:",
    CANDIDATE_CSV
)

print(
    "JSONL:",
    CANDIDATE_JSONL
)

CSV: D:\dev\projects\fourlang_translation\data\clean\en_uz\hplt\hplt_en_uz_10k_clean.csv
JSONL: D:\dev\projects\fourlang_translation\data\clean\en_uz\hplt\hplt_en_uz_10k_clean.jsonl


In [50]:
save_df[
    "document_group"
] = (
    save_df[
        "src_doc_id"
    ].astype(str)
    +
    "||"
    +
    save_df[
        "tgt_doc_id"
    ].astype(str)
)

In [51]:
def assign_split(
    group_key
):
    digest = hashlib.md5(
        group_key.encode(
            "utf-8"
        )
    ).hexdigest()

    number = int(
        digest[:8],
        16
    )

    bucket = number % 100

    if bucket < 90:
        return "train"

    elif bucket < 95:
        return "validation"

    else:
        return "test"

In [52]:
save_df["split"] = (
    save_df[
        "document_group"
    ].apply(
        assign_split
    )
)

In [53]:
save_df[
    "split"
].value_counts()

split
train         8980
validation     528
test           492
Name: count, dtype: int64

In [54]:
group_split_count = (
    save_df
    .groupby(
        "document_group"
    )["split"]
    .nunique()
)

print(
    "跨split document数量:",
    (
        group_split_count > 1
    ).sum()
)

跨split document数量: 0


In [55]:
SPLIT_DIR = (
    CLEAN_DIR
    / "splits"
)

SPLIT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [56]:
for split_name in [
    "train",
    "validation",
    "test",
]:

    split_df = save_df[
        save_df["split"]
        ==
        split_name
    ].copy()

    output_file = (
        SPLIT_DIR
        / f"{split_name}.jsonl"
    )

    split_df.to_json(
        output_file,
        orient="records",
        lines=True,
        force_ascii=False,
    )

    print(
        split_name,
        len(split_df),
        "→",
        output_file
    )

train 8980 → D:\dev\projects\fourlang_translation\data\clean\en_uz\hplt\splits\train.jsonl
validation 528 → D:\dev\projects\fourlang_translation\data\clean\en_uz\hplt\splits\validation.jsonl
test 492 → D:\dev\projects\fourlang_translation\data\clean\en_uz\hplt\splits\test.jsonl


In [58]:
def make_bidirectional(
    pair_df
):
    forward = pd.DataFrame({
        "pair_id":
            pair_df["pair_id"],

        "src_lang":
            "en",

        "tgt_lang":
            "uz",

        "src_text":
            pair_df["en"],

        "tgt_text":
            pair_df["uz"],

        "direction":
            "en-uz",

        "split":
            pair_df["split"],
    })

    backward = pd.DataFrame({
        "pair_id":
            pair_df["pair_id"],

        "src_lang":
            "uz",

        "tgt_lang":
            "en",

        "src_text":
            pair_df["uz"],

        "tgt_text":
            pair_df["en"],

        "direction":
            "uz-en",

        "split":
            pair_df["split"],
    })

    result = pd.concat(
        [
            forward,
            backward,
        ],
        ignore_index=True,
    )

    return result

In [59]:
directional_df = (
    make_bidirectional(
        save_df
    )
)

In [60]:
print(
    "Parallel pairs:",
    len(save_df)
)

print(
    "Directional samples:",
    len(directional_df)
)

Parallel pairs: 10000
Directional samples: 20000


In [61]:
directional_df[
    "direction"
].value_counts()

direction
en-uz    10000
uz-en    10000
Name: count, dtype: int64

In [62]:
DIRECTION_DIR = (
    CLEAN_DIR
    / "directional"
)

DIRECTION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [63]:
for split_name in [
    "train",
    "validation",
    "test",
]:

    split_df = directional_df[
        directional_df["split"]
        ==
        split_name
    ].copy()

    output_file = (
        DIRECTION_DIR
        / f"{split_name}.jsonl"
    )

    split_df.to_json(
        output_file,
        orient="records",
        lines=True,
        force_ascii=False,
    )

    print(
        split_name,
        len(split_df),
        "→",
        output_file
    )

train 17960 → D:\dev\projects\fourlang_translation\data\clean\en_uz\hplt\directional\train.jsonl
validation 1056 → D:\dev\projects\fourlang_translation\data\clean\en_uz\hplt\directional\validation.jsonl
test 984 → D:\dev\projects\fourlang_translation\data\clean\en_uz\hplt\directional\test.jsonl


In [64]:
train_path = (
    DIRECTION_DIR
    / "train.jsonl"
)

train_df = pd.read_json(
    train_path,
    lines=True,
)

train_df.sample(
    20,
    random_state=42
)

,pair_id,src_lang,tgt_lang,src_text,tgt_text,direction,split
7075,hplt_00007981,en,uz,[40] They were criticized for not doing safety reasons.,[40] Ular xavfsizlik sabablari qilmagan uchun keskin tanqid qilindi.,en-uz,train
13167,hplt_00004788,uz,en,"Agar siz ilgari uyingizni quyon bilan hech qachon baham ko'rmagan bo'lsangiz, bilishingiz kerak bo'lgan ba'zi narsalar mavjud.","If you have never shared your home with a rabbit before, there are some things you need to know.",uz-en,train
13936,hplt_00005619,uz,en,Bolalar bog'chasi haqidagi ertak va hikoyalar,Fairy tales and stories about kindergarten,uz-en,train
12405,hplt_00003844,uz,en,Mopedga onlayn buyurtma berish mumkin va butun dunyo bo'ylab etkazib beriladi.,The moped can be ordered online and is shipped worldwide.,uz-en,train
2325,hplt_00002609,en,uz,"You see that I have had a very serious gambling- and alcohol problem, which has deteriorated in the past 10 arene.","Siz men juda jiddiy edi, deb ko'rish qimor- va spirtli muammo, qaysi o'tmishda yomonlashdi 10 Mfntvfnbr.",en-uz,train
7700,hplt_00008649,en,uz,"A popular wall solution is to choose a console, which is both a shelf and a part of the wall for the TV in which it is mounted.","Ommabop devor yechimi - bu konsolni tanlashdir, u ham rafga, ham o'rnatilgan telekanal uchun devorning bir qismiga to'g'ri keladi.",en-uz,train
17746,hplt_00009781,uz,en,Oshxonani operatsiya xonasiga aylantirmaslik uchun xonaga rang urg'u qo'shing.,"In order not to turn the kitchen into the operating room, add color accents to the room.",uz-en,train
12239,hplt_00003657,uz,en,Multifaktorial autentifikatsiya sizning LastPass hisobi bilan foydalanish uchun faollashtirilgan bo'lishi mumkin bo'lgan qurilma anglatadi va siz hisobingizga kirish uchun avval ikkinchi qadam talab.,Multifactor authentication refers to a device that can be enabled for use with your LastPass account and requires a second step before you can gain access to your account.,uz-en,train
16711,hplt_00008685,uz,en,Studiya kvartirasi egalari odatda bunday echimlarga ega bo'lgan yashash xonasini zabt etishadi.,Studio apartment owners often zone a living room with such solutions.,uz-en,train
5526,hplt_00006245,en,uz,So this will basically be messages from you at the end of June.,"Shunday qilib, bu asosan iyun oyining oxirida sizdan keladigan xabarlar bo'ladi.",en-uz,train


In [65]:
license_record = {
    "resource":
        "HPLT/DocHPLT",

    "subset":
        "en-uz",

    "revision":
        REVISION,

    "type":
        "parallel_translation",

    "dataset_card_license":
        "CC0-1.0",

    "commercial_status":
        "candidate_requires_internal_review",

    "usage":
        "M2M100 en-uz fine-tuning experiment",

    "pairs_extracted":
        int(
            len(save_df)
        ),
}

In [66]:
LICENSE_INFO_FILE = (
    DOCS_DIR
    / "hplt_en_uz_dataset_info.json"
)

with open(
    LICENSE_INFO_FILE,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        license_record,
        f,
        ensure_ascii=False,
        indent=2,
    )

print(
    LICENSE_INFO_FILE
)

D:\dev\projects\fourlang_translation\docs\hplt_en_uz_dataset_info.json


In [67]:
summary = {
    "dataset":
        DATASET_ID,

    "config":
        CONFIG_NAME,

    "revision":
        REVISION,

    "target_pairs":
        TARGET_PAIRS,

    "final_pairs":
        int(
            len(save_df)
        ),

    "directional_samples":
        int(
            len(directional_df)
        ),

    "aligner_min":
        ALIGNER_MIN,

    "bicleaner_min":
        BICLEANER_MIN,

    "max_chars":
        MAX_CHARS,

    "train_pairs":
        int(
            (
                save_df["split"]
                == "train"
            ).sum()
        ),

    "validation_pairs":
        int(
            (
                save_df["split"]
                == "validation"
            ).sum()
        ),

    "test_pairs":
        int(
            (
                save_df["split"]
                == "test"
            ).sum()
        ),

    "extraction_stats":
        dict(
            extraction_stats
        ),
}

In [68]:
SUMMARY_FILE = (
    RESULT_DIR
    / "hplt_en_uz_10k_summary.json"
)

with open(
    SUMMARY_FILE,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        summary,
        f,
        ensure_ascii=False,
        indent=2,
    )

print(
    SUMMARY_FILE
)

D:\dev\projects\fourlang_translation\results\hplt_en_uz_10k_summary.json


In [69]:
print("=" * 70)
print("HPLT en-uz Dataset Check")
print("=" * 70)

print(
    "Pairs:",
    len(save_df)
)

print(
    "Directional samples:",
    len(directional_df)
)

print(
    "Duplicates:",
    save_df.duplicated(
        subset=[
            "en",
            "uz",
        ]
    ).sum()
)

print(
    "Empty EN:",
    (
        save_df["en"]
        .str.strip()
        ==
        ""
    ).sum()
)

print(
    "Empty UZ:",
    (
        save_df["uz"]
        .str.strip()
        ==
        ""
    ).sum()
)

print(
    "Cyrillic Uzbek:",
    save_df[
        "uz"
    ].apply(
        has_cyrillic
    ).sum()
)

print(
    "Benchmark leakage:",
    len(leak_df)
)

print()
print("Split:")

print(
    save_df[
        "split"
    ].value_counts()
)

print()
print("Direction:")

print(
    directional_df[
        "direction"
    ].value_counts()
)

HPLT en-uz Dataset Check
Pairs: 10000
Directional samples: 20000
Duplicates: 0
Empty EN: 0
Empty UZ: 0
Cyrillic Uzbek: 0
Benchmark leakage: 0

Split:
split
train         8980
validation     528
test           492
Name: count, dtype: int64

Direction:
direction
en-uz    10000
uz-en    10000
Name: count, dtype: int64


In [70]:
pairs_df[
    [
        "aligner_score",
        "bicleaner_score",
        "bifixer_score",
    ]
].describe(
    percentiles=[
        0.01,
        0.05,
        0.10,
        0.25,
        0.50,
        0.75,
        0.90,
        0.95,
        0.99,
    ]
)

,aligner_score,bicleaner_score,bifixer_score
count,10000.000000,10000.000000,10000.000000
mean,0.645378,0.982706,1.056220
std,0.122493,0.034294,0.681306
min,0.500000,0.800000,0.709000
1%,0.500025,0.826000,0.807798
5%,0.507117,0.906000,0.897600
10%,0.517103,0.946000,0.914390
25%,0.550481,0.984000,0.933400
50%,0.615886,0.998000,0.950100
75%,0.707107,1.000000,0.969600


In [71]:
print(
    "最低 aligner:",
    pairs_df["aligner_score"].min()
)

print(
    "最低 bicleaner:",
    pairs_df["bicleaner_score"].min()
)

print(
    "低于阈值数量:",
    (
        (pairs_df["aligner_score"] < ALIGNER_MIN)
        |
        (pairs_df["bicleaner_score"] < BICLEANER_MIN)
    ).sum()
)

最低 aligner: 0.5
最低 bicleaner: 0.800000011920929
低于阈值数量: 0


In [72]:
borderline_df = (
    pairs_df.sort_values(
        [
            "bicleaner_score",
            "aligner_score",
        ],
        ascending=True,
    )[
        [
            "en",
            "uz",
            "aligner_score",
            "bicleaner_score",
            "bifixer_score",
        ]
    ]
    .head(50)
)

borderline_df

,en,uz,aligner_score,bicleaner_score,bifixer_score
1098,Instagram would your photos to third parties (Advertisers) could sell.,Instagram uchinchi shaxslarga sizning rasmlaringiz edi (Reklama beruvchilar) sotish mumkin.,0.506419,0.800,0.9645
2270,Can you have surgery on your fish?,Baliqingizda operatsiya qilolmaysizmi?,0.566947,0.800,0.9817
383,Want to import this vehicle to Australia?,Ushbu transport vositasini Uzbekistan-ga import qilmoqchimisiz?,0.619628,0.800,0.9837
9238,- purify the air;,- Havoni tozalash;,0.632456,0.800,0.9026
430,Other mechanisms transport much larger molecules.,Boshqa mexanizmlar ancha katta molekulalarni tashiydi.,0.755929,0.800,0.9828
7409,- purchase of bonds of the Central Bank of the Republic of Uzbekistan;,- Foizi - O’zbekiston Respublikasi Markaziy banki qayta moliyalashtirish stavkasining 50 foizi;,0.550225,0.801,1.4232
5728,Structures were erected with cast iron and wrought iron frames.,Konstruksiyalar quyma temir va temir ramkalar bilan qurilgan.,0.564076,0.801,0.9722
5852,"Next, you will go Waisai, capital of Raja Ampat.","Keyingi, siz Waisai, Raja Ampat poytaxti ketadi.",0.724071,0.801,0.9001
5534,- You can chat via your mobile device or through your laptop.,Siz mobil qurilmangiz yoki noutbuk orqali suhbatlashishingiz mumkin.,0.581001,0.802,0.9622
3149,The battery has a trolley system that makes it easy to transport the battery.,Batareyada akkumulyatorni tashish osonlashtiradigan tramvay tizimi mavjud.,0.786796,0.802,0.9769


In [73]:
best_df = (
    pairs_df.sort_values(
        [
            "bicleaner_score",
            "aligner_score",
        ],
        ascending=False,
    )[
        [
            "en",
            "uz",
            "aligner_score",
            "bicleaner_score",
            "bifixer_score",
        ]
    ]
    .head(30)
)

best_df

,en,uz,aligner_score,bicleaner_score,bifixer_score
15,"After that, follow the instructions sent from the service to your email address.","Shundan so'ng, xizmatdan elektron pochta manzilingizga yuborilgan ko'rsatmalarga amal qiling.",1.0,1.0,0.9619
477,Directions and map.,Yo'nalishlari va xaritasi.,1.0,1.0,0.9500
842,It depends on the terms and conditions of nicovideo.jp.,Bu nicovideo.jp shartlari va shartlariga bog'liq.,1.0,1.0,0.9421
1331,We work with more than 20 payment systems.,Biz 20 dan ortiq to'lov tizimlari bilan ishlaymiz.,1.0,1.0,0.9224
1666,English at the level of reading and understanding of technical literature,Texnik adabiyotlarni o'qish va tushunish darajasida ingliz tili,1.0,1.0,0.9664
1677,"Your city, stat, zip code.","sizning shahar, stat, zip kodi.",1.0,1.0,0.9118
1997,The scooter can be ordered in many color combinations.,Skuterga ko'plab rang kombinatsiyalarida buyurtma berish mumkin.,1.0,1.0,0.9620
2375,"Unlike air conditioning equipment, induction heating does not create an ideal environment for bacteria to grow.","Konditsioner uskunasidan farqli o'laroq, induksion isitish bakteriyalarning ko'payishi uchun ideal muhit yaratmaydi.",1.0,1.0,0.9696
2417,The scooter is available in many colors and can be ordered online.,Skuter juda ko'p ranglarda mavjud va uni onlayn buyurtma qilish mumkin.,1.0,1.0,0.9409
2841,"No, you can use this tool immediately after opening this page.","Yo'q, ushbu sahifani ochgandan so'ng darhol ushbu vositadan foydalanishingiz mumkin.",1.0,1.0,0.9473


In [74]:
pairs_df["uz_has_cyrillic"] = (
    pairs_df["uz"]
    .apply(has_cyrillic)
)

pairs_df[
    "uz_has_cyrillic"
].value_counts()

uz_has_cyrillic
False    10000
Name: count, dtype: int64

In [75]:
same_text_df = pairs_df[
    pairs_df["en"]
    .apply(normalize_for_match)
    ==
    pairs_df["uz"]
    .apply(normalize_for_match)
]

print(
    "两边文本完全一致:",
    len(same_text_df)
)

same_text_df.head(30)

两边文本完全一致: 0


,pair_id,en,uz,en_sentence_id,uz_sentence_id,src_doc_id,tgt_doc_id,lang_pair,aligner_score,bicleaner_score,bifixer_score,dataset,dataset_revision,en_chars,uz_chars,uz_has_cyrillic,en_normalized,uz_normalized
